# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandhanamj/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
%pip -q install duckdb

In [22]:
import os
import duckdb

con = duckdb.connect()

print("DuckDB connection ready.")

DuckDB connection ready.


In [23]:
# Connect DuckDB to the gated Hugging Face warehouse.
# The token is read from Colab Secrets, so the secret is NOT stored in this notebook.

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face authentication ready.")

Hugging Face authentication ready.


In [24]:
# Use the March 2026 partition for development.
# The assignment specifically recommends a mid-panel month instead of the final June 2026 month.
# The final month is treated as a sealed test window.

PERF_MARCH = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet'"
    ")"
)

print("March 2026 performance partition ready.")

March 2026 performance partition ready.


## 1. Unit of analysis + time window

### Unit of analysis

One row represents one content item for one client on one report date in the daily performance warehouse.

### Tables used

The main table is `fact_content_daily_performance`. The March 2026 partition is used for the feature/decision window, and the April 2026 partition is used only to construct the future outcome label.

### Time window

The feature and decision window is March 1–31, 2026. The future outcome window is April 1–30, 2026. June 2026 is treated as the sealed final month and is not used for development.

### What to predict / rank

The content-refresh lane ranks content items using a future click-decline proxy. An item is labeled `1` when its total GSC clicks in April 2026 are lower than its total GSC clicks in March 2026.

### Deliberately excluded

Future performance and any label-derived field are deliberately excluded from the model features because they would not be available at the March decision point.

In [25]:
# Inspect the actual columns in the March 2026 performance partition.
# Do this before writing verification queries so we do not guess field names.

schema_check = con.sql(f"""
    DESCRIBE SELECT *
    FROM {PERF_MARCH}
""").df()

schema_check[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields

**Five initial features**

- `gsc_impressions` — knowable at the decision moment because it is an observed search-visibility measure from the feature window.
- `gsc_clicks` — knowable at the decision moment because it is an observed search-click measure from the feature window.
- `gsc_avg_position` — knowable at the decision moment because it is an observed average search-position measure from the feature window.
- `ga4_pageviews` — knowable at the decision moment because it is an observed pageview measure from the feature window when GA4 data is available.
- `ga4_engaged_sessions` — knowable at the decision moment because it is an observed engaged-session measure from the feature window when GA4 data is available.

**Label / proxy**

The target is whether a content item shows a decline in performance in a future observation window. This future outcome is used for refresh-prioritization ranking and is not available at the decision moment.

**Context**

- `report_date` — identifies when the observation was recorded and supports time-based splitting.
- `client_hash_id` — identifies the pseudonymized client for grouping and splitting; it is not a model feature.
- `content_hash_id` — identifies the pseudonymized content item; it is not a model feature.

**Excluded**

Future performance and any field derived from the future outcome are excluded because they would reveal information that would not be available when deciding which content to refresh.

In [26]:
# Define the five initial features for the content-refresh lane.
# These are measured performance signals available before the decision point.

FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions",
]

print("Five initial features:")
for feature in FEATURES:
    print("-", feature)

Five initial features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions


In [27]:
# Build a small feature frame from March 2026.
# We keep the data at the warehouse grain:
# one content item for one client on one report date.
#
# Only the five selected features are brought back to pandas.

feature_frame = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_engaged_sessions
    FROM {PERF_MARCH}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    LIMIT 10000
""").df()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (10000, 8)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>


### Decision point and future label

The decision point is the end of March 2026. The five features are measured during March 2026 and represent information available before the refresh-prioritization decision.

For this first warehouse experiment, the label/proxy is **future click decline**: an item is labeled `1` when its total GSC clicks in April 2026 are lower than its total GSC clicks in March 2026.

April is treated as the future outcome window, so April performance is never used as a model feature.

This is a simple directional proxy for refresh prioritization, not proof that a refresh would improve performance.

In [28]:
# Build a simple future-outcome proxy.
#
# March is the feature/decision window.
# April is the future outcome window.
# The label is 1 when April GSC clicks are lower than March GSC clicks.

PERF_APRIL = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/*.parquet'"
    ")"
)

label_frame = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS march_clicks
        FROM {PERF_MARCH}
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks
        FROM {PERF_APRIL}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.march_clicks,
        a.april_clicks,
        CASE
            WHEN a.april_clicks < m.march_clicks THEN 1
            ELSE 0
        END AS future_click_decline
    FROM march m
    INNER JOIN april a
        ON m.client_hash_id = a.client_hash_id
       AND m.content_hash_id = a.content_hash_id
    LIMIT 10000
""").df()

print("Label frame shape:", label_frame.shape)
label_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Label frame shape: (10000, 5)


,client_hash_id,content_hash_id,march_clicks,april_clicks,future_click_decline
0,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,1.0,1.0,0
1,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,2.0,0.0,1
2,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,1.0,0.0,1
3,client_62f4a7e64f5e0096,content_d47ba5533f9c8573,0.0,0.0,0
4,client_62f4a7e64f5e0096,content_50266f97d6233542,0.0,0.0,0


### Leakage trap

To demonstrate leakage, I deliberately add the label-derived field `future_click_decline` to the feature set and measure how directly it predicts itself.

This is intentionally invalid: the field is the answer we are trying to predict, so it would not be available at the March decision point.

The leaked result is shown only as a sanity check. The leaked field is then removed and is not retained as a feature.

In [29]:
# Deliberate leakage experiment.
# The label itself is added as a "feature" so we can see the score jump to perfect.
# This is intentionally invalid and will be removed immediately afterward.

leaky_features = label_frame[["future_click_decline"]].copy()
leaky_target = label_frame["future_click_decline"]

leaky_accuracy = (
    leaky_features["future_click_decline"] == leaky_target
).mean()

print(f"Leaky accuracy: {leaky_accuracy:.3f}")

Leaky accuracy: 1.000


### Honest feature set after leakage check

The deliberate leak is removed. The final feature set contains only the five March 2026 performance signals defined earlier.

`future_click_decline`, `march_clicks`, and `april_clicks` are not model features because they contain or directly support the future outcome.

In [30]:
# Remove the deliberately leaked label-derived field.
# Keep only the five features defined in the contract.

honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions",
]

honest_feature_frame = feature_frame[
    ["report_date", "client_hash_id", "content_hash_id"] + honest_features
].copy()

print("Honest feature columns:")
for feature in honest_features:
    print("-", feature)

print("\nFeature frame shape:", honest_feature_frame.shape)
honest_feature_frame.head()

Honest feature columns:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions

Feature frame shape: (10000, 8)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>


## 3. Verify it with queries

*The three checks below verify the warehouse grain, March 2026 slice, and data availability.*

In [31]:
# Verify the grain:
# Each combination of report_date, client_hash_id, and content_hash_id
# should appear at most once in the daily performance table.
#
# If this returns zero rows, the claimed grain holds for March 2026.

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {PERF_MARCH}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate grain combinations found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,report_date,client_hash_id,content_hash_id,row_count


In [32]:
# Verification 2: confirm the number of March 2026 rows
# and the observed date range in the development partition.

march_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {PERF_MARCH}
""").df()

march_summary

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [33]:
# Verification 3: show a sample of March 2026 rows where both
# Google Search Console and Google Analytics 4 data are available.
# IS TRUE is used explicitly so only rows marked as available survive.

availability_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        ga4_data_available,
        COUNT(*) OVER () AS available_row_count
    FROM {PERF_MARCH}
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    LIMIT 5
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_data_available,ga4_data_available,available_row_count
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,True,True,364347
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,True,True,364347
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,True,True,364347
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,True,True,364347
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,True,True,364347


## 4. Data limits

This warehouse supports decision-support ranking, but it has important limits.

- History is not equally deep for every client, so older observations may not be comparable across all clients.
- Some rows have GSC data while GA4 data is unavailable. The feature frame therefore should not assume that a missing GA4 value means zero activity.
- The March 2026 development window is observed data, while a future-window outcome would require later observations. The feature and label windows must remain separated to avoid leakage.
- The pseudonymized client and content IDs are identifiers for grouping, joining, and splitting, not meaningful performance features.
- This analysis can identify observed performance patterns associated with refresh priority, but it cannot establish why a page declined or whether a refresh would cause improvement.

**Named limitation:** GA4 availability is uneven, so a ranking based on all five features may apply to a smaller subset of rows than a GSC-only analysis.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, private URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.